# 1. JAAT-coded postings → occupation-task profiles

Use all available coded USAJOBS postings to count how often each task appears in each occupation. We start from saved TitleMatch occupation codes and TaskMatch task IDs; we do **not** rerun historical neural inference. A task counts once per posting, even if retrieved several times.

The reference run contains 3,089,366 unique posting IDs, with start dates January 2017–March 2026. This is an assembled corpus, not a claim of complete coverage of every USAJOBS announcement. These public-source profiles are distinct from the paper's sampled NLx profiles.

Run `pip install -r requirements.txt` in your notebook environment. The configuration cell downloads pinned public [coded postings](https://huggingface.co/datasets/loyoladatamining/usajobs_coded) and [date metadata](https://huggingface.co/datasets/loyoladatamining/usajobs_html_metadata) if the input folders are absent (about 0.85 GB). No Hugging Face account or token is needed. Extract the complete notebook package first; it includes `download_public_inputs.py` and reference data. You can instead set `INPUT` and `META` to existing monthly folders. Notebook 2 includes aggregate inputs and runs independently.

**Choices:** all dates; one supplied occupation assignment per posting; no new title-score cutoff; unique posting–task pairs; O*NET 27.2 task IDs and 31.0 occupation-code eligibility, matching the existing USAJOBS analysis. No two-LLM review ledger is applied here; do not describe this as reproducing the NLx review procedure.

In [1]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
DATA = Path("data")
OUT = Path("results")
OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 100)

In [2]:
# Edit this path. An environment override is optional and used only for local testing.
INPUT = Path(os.environ.get("USAJOBS_CODED_INPUT", "usajobs_coded/postings"))  # hf.co/datasets/loyoladatamining/usajobs_coded
META = Path(os.environ.get("USAJOBS_META_INPUT", "usajobs_html_metadata/postings"))  # hf.co/datasets/loyoladatamining/usajobs_html_metadata
from download_public_inputs import download, DATASETS
if not INPUT.exists(): download('coded', INPUT)
if not META.exists(): download('meta', META)
files = sorted(INPUT.glob("*.parquet")) if INPUT.is_dir() else [INPUT]
if not files or any(not p.exists() for p in files):
    raise FileNotFoundError("Set INPUT to the saved JAAT-coded Parquet file or monthly directory.")
import duckdb
con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='4GB'")
meta_files = sorted(META.glob("*.parquet"))
if not meta_files:
    raise FileNotFoundError("Set META to the usajobs_html_metadata monthly directory (start_date/end_date).")
# Public coded files key postings by usajobsControlNumber and omit dates; dates live in the companion metadata.
con.execute(f"""CREATE VIEW raw AS SELECT CAST(c.usajobsControlNumber AS BIGINT) AS job_id, m.start_date, m.end_date, c.*
 FROM read_parquet({[str(p) for p in files]}) c
 LEFT JOIN read_parquet({[str(p) for p in meta_files]}) m ON CAST(c.usajobsControlNumber AS BIGINT) = m.usajobsControlNumber""")
required = {"job_id", "start_date", "end_date", "title_code", "title_score", "tasks"}
assert required <= set(con.table("raw").columns), "Input schema differs from saved JAAT records"
# No date filter and no random sampling: every supplied posting enters this cohort.
con.execute("CREATE VIEW cohort AS SELECT * FROM raw")
display(con.execute("SELECT job_id, title_code, title_score, tasks FROM cohort ORDER BY job_id LIMIT 5").df())

,job_id,title_code,title_score,tasks
0,309472900,15-1231.00,0.937,3464;18103;680;22626;17215;2690;21784;3481
1,309482600,15-1231.00,0.937,3464;18103;680;22626;17215;2690;21784;3481
2,314527500,13-1199.00,0.920,15258;3774;21426;22232;3774
3,314799200,13-1199.00,0.920,15258;3774;21426;3774
4,316399800,29-1224.00,0.895,4110;10748;18449;19912;2828;17239;2043;17482;22735;3744;10748;17231;20055;17699


## Check the unit and source coverage

Repeated posting IDs are an error requiring investigation, not automatically dropped records. Missing tasks remain in occupation posting totals. Date anomalies are reported; because this demonstration uses all records, dates do not select the sample.

The pinned public metadata yield 17,348 end-before-start records. The historical local input had 10,674 because some end dates were missing. Dates do not select records in this all-postings run, so the rebuilt profiles are unchanged.

In [3]:
diagnostics = con.execute("""SELECT count(*) AS rows,
 count(DISTINCT job_id) AS unique_ids,
 count(*) FILTER (WHERE job_id IS NULL) AS missing_ids,
 min(start_date) AS earliest_start, max(start_date) AS latest_start,
 count(*) FILTER (WHERE end_date < start_date) AS end_before_start,
 count(*) FILTER (WHERE tasks IS NULL OR trim(tasks)='') AS empty_task_strings,
 count(*) FILTER (WHERE title_code IS NULL OR trim(title_code)='') AS missing_codes
 FROM cohort""").df()
display(diagnostics)
assert diagnostics.rows.iloc[0] == diagnostics.unique_ids.iloc[0], "Repeated or missing posting IDs"
assert diagnostics.missing_ids.iloc[0] == 0
con.execute("""CREATE TEMP VIEW tokens AS
 SELECT job_id, title_code AS code, trim(token) AS token
 FROM cohort, unnest(string_split(coalesce(tasks,''),';')) t(token)""")
parse_errors = con.execute("""SELECT token, count(*) AS occurrences FROM tokens
 WHERE token<>'' AND NOT regexp_full_match(token,'[0-9]+') GROUP BY token""").df()
display(parse_errors)
parse_errors.to_csv(OUT/'malformed_task_tokens.csv', index=False)
# Mirror the original parser: accept integer tokens, excluding malformed tokens.
con.execute("""CREATE TEMP VIEW pairs AS
 SELECT DISTINCT job_id,code,try_cast(token AS BIGINT) AS task_id
 FROM tokens WHERE regexp_full_match(token,'[0-9]+')""")
print('Repeated valid retrievals removed:', con.execute("""SELECT
 (SELECT count(*) FROM tokens WHERE regexp_full_match(token,'[0-9]+'))
 - (SELECT count(*) FROM pairs)""").fetchone()[0])

,rows,unique_ids,missing_ids,earliest_start,latest_start,end_before_start,empty_task_strings,missing_codes
0,3089366,3089366,0,2017-01-01,2026-03-31,17348,39453,0


,token,occurrences


Repeated valid retrievals removed: 5076639


## Count postings and task incidences

Every posting has weight 1 because we use all supplied postings. The commented example below indicates where inverse-probability weights would enter **if** observations had instead been sampled. Unequal occupation sizes alone do not justify inverse weights. This unit is a posting ID, not an active posting-month.

In [4]:
# For a genuinely sampled design only:
# posting_weight = 1 / inclusion_probability_for_occupation_month
# Then join that weight to unique posting-task pairs, sum weights for both
# occupation totals and task incidences, and keep raw counts for support filtering.
# Do not activate this for the present all-postings corpus.
totals = con.execute("""SELECT title_code AS code,count(*) AS n_documents,
 count(*)::DOUBLE AS weight,0 AS shared_documents FROM cohort GROUP BY 1 ORDER BY 1""").df()
incidence = con.execute("""SELECT code,task_id,count(*) AS n_documents,
 count(*)::DOUBLE AS weight FROM pairs GROUP BY 1,2 ORDER BY 1,2""").df()
assert np.all(incidence.weight == incidence.n_documents)
assert np.all(incidence.n_documents <= incidence.code.map(totals.set_index('code').n_documents))
totals.to_parquet(OUT/'totals8.parquet',index=False)
incidence.to_parquet(OUT/'incidence8.parquet',index=False)
display(totals.sort_values('n_documents',ascending=False).head(10))

,code,n_documents,weight,shared_documents
80,13-1111.00,121414,121414.0,0
114,15-1232.00,79351,79351.0,0
73,13-1071.00,67910,67910.0,0
672,43-6011.00,64017,64017.0,0
433,29-1141.00,59138,59138.0,0
513,31-9092.00,49407,49407.0,0
95,13-2031.00,46294,46294.0,0
23,11-3071.00,41943,41943.0,0
62,13-1023.00,40450,40450.0,0
675,43-6014.00,38388,38388.0,0


## Inspect the occupation-task profiles

Keep a link if it appears in at least five distinct postings. The default corpus-wide RCA reference in Notebook 2 uses the **retained** links, matching the paper. The earlier pre-filter USAJOBS reference remains an optional sensitivity. No link is dropped merely because O*NET assigns that task to another occupation.

Posting prevalence is the fraction of an occupation's postings containing a task. Task-incidence share has a different denominator: all retained task incidences in that occupation. RCA below uses incidence shares.

In [5]:
tasks = pd.read_csv(DATA/'tasks.csv')
occupations = pd.read_csv(DATA/'occupations.csv')
valid = incidence.code.isin(occupations['O*NET-SOC Code']) & incidence.task_id.isin(tasks.task_id)
invalid = incidence.loc[~valid].copy()
invalid.to_csv(OUT/'invalid_occupation_task_counts.csv',index=False)
print('Excluded incidence rows with invalid code or task ID:',len(invalid))
profiles = incidence.loc[valid & incidence.n_documents.ge(5)].copy()
profiles['posting_prevalence'] = profiles.weight / profiles.code.map(totals.set_index('code').weight)
profiles['task_incidence_share'] = profiles.weight / profiles.groupby('code').weight.transform('sum')
profiles['task_text'] = profiles.task_id.map(tasks.set_index('task_id').task_text)
canonical = set(zip(tasks.occ8,tasks.task_id))
profiles['assigned_by_onet'] = [(c,t) in canonical for c,t in zip(profiles.code,profiles.task_id)]
profiles.to_parquet(OUT/'retained_profiles.parquet',index=False)
for c in ['29-1141.00','11-9111.00']:
    print(c, occupations.set_index('O*NET-SOC Code').loc[c,'Title'])
    display(profiles[profiles.code.eq(c)].sort_values('weight',ascending=False).head(15))
# Prove agreement with the aggregate inputs shipped for Notebook 2.
for name,frame in [('totals8',totals),('incidence8',incidence)]:
    expected = pd.read_parquet(DATA/(name+'.parquet'))
    keys=['code'] if name=='totals8' else ['code','task_id']
    pd.testing.assert_frame_equal(frame.sort_values(keys).reset_index(drop=True),
        expected.sort_values(keys).reset_index(drop=True),check_dtype=False)
print('PASS: rebuilt counts equal the saved full-corpus analysis.')

Excluded incidence rows with invalid code or task ID: 0


29-1141.00 Registered Nurses


,code,task_id,n_documents,weight,posting_prevalence,task_incidence_share,task_text,assigned_by_onet
483964,29-1141.00,2699,43523,43523.0,0.735957,0.095045,Select applicants meeting specified job requirements and refer them to hiring personnel.,False
486259,29-1141.00,18855,18831,18831.0,0.318425,0.041123,Conduct reference or background checks on job applicants.,False
486274,29-1141.00,18871,17754,17754.0,0.300213,0.038771,"Select qualified job applicants or refer them to managers, making hiring recommendations when ap...",False
483845,29-1141.00,2016,15025,15025.0,0.254067,0.032811,"Transport patients to and from treatment areas, lifting and transferring them according to posit...",False
485642,29-1141.00,17067,12969,12969.0,0.219301,0.028322,Conduct physical examinations of patients.,False
486222,29-1141.00,18699,10841,10841.0,0.183317,0.023674,Communicate and collaborate with other healthcare professionals involved with the care of a pati...,False
485661,29-1141.00,17098,7181,7181.0,0.121428,0.015682,"Direct, coordinate, or supervise the patient care activities of nursing or support staff.",False
483752,29-1141.00,1742,7079,7079.0,0.119703,0.015459,"Plan, organize, and conduct practice sessions.",False
486478,29-1141.00,20055,6497,6497.0,0.109862,0.014188,"Administer drug and alcohol tests, including random drug screens of offenders, to verify complia...",False
483850,29-1141.00,2027,5689,5689.0,0.096199,0.012424,Prepare and administer medications as directed by a physician.,False


11-9111.00 Medical and Health Services Managers


,code,task_id,n_documents,weight,posting_prevalence,task_incidence_share,task_text,assigned_by_onet
66299,11-9111.00,2699,22566,22566.0,0.694210,0.082128,Select applicants meeting specified job requirements and refer them to hiring personnel.,False
68781,11-9111.00,18855,11680,11680.0,0.359318,0.042509,Conduct reference or background checks on job applicants.,False
68794,11-9111.00,18871,10353,10353.0,0.318495,0.037679,"Select qualified job applicants or refer them to managers, making hiring recommendations when ap...",False
68204,11-9111.00,17067,5625,5625.0,0.173045,0.020472,Conduct physical examinations of patients.,False
66184,11-9111.00,2016,4626,4626.0,0.142312,0.016836,"Transport patients to and from treatment areas, lifting and transferring them according to posit...",False
68757,11-9111.00,18699,3561,3561.0,0.109549,0.012960,Communicate and collaborate with other healthcare professionals involved with the care of a pati...,False
68606,11-9111.00,18353,3107,3107.0,0.095582,0.011308,Supervise and monitor unit nursing staff.,False
69378,11-9111.00,21339,2815,2815.0,0.086599,0.010245,Supervise subordinate employees.,False
68640,11-9111.00,18404,2724,2724.0,0.083800,0.009914,Supervise or coordinate patient care or support staff activities.,False
67386,11-9111.00,9738,2637,2637.0,0.081123,0.009597,Check with employers or other references to verify answers and obtain further information.,False


PASS: rebuilt counts equal the saved full-corpus analysis.


In [6]:
# Input fingerprints make a different corpus detectable without redistributing it.
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
manifest={'inputs':[{'file':p.name,'sha256':sha256(p)} for p in files],
 'metadata_inputs':[{'file':p.name,'sha256':sha256(p)} for p in meta_files],
 'public_dataset_revisions':DATASETS,
 'unit':'unique posting ID','sampling_weight':1,'support':5,
 'review':'No occupation-task LLM review ledger applied',
 'diagnostics':diagnostics.astype(str).to_dict('records')}
(OUT/'profile_run.json').write_text(json.dumps(manifest,indent=2))
con.close()
print('Next: open Notebook 2.')

Next: open Notebook 2.
